# CURATE.AI's randomized trial: what did it actually test?

Khong et al., *Nature Communications* 2025, "Tacrolimus dosing in liver transplant recipients using phenotypic personalized medicine: a phase 2 randomized clinical trial",
doi 10.1038/s41467-025-59739-6, CC BY, NCT03527238. 62 randomized, 56 analysed (29 standard of care, 27 "PPM", the CURATE.AI group's method).
Primary outcome: % of post-transplant days with trough level more than 2 ng/mL outside the target range. Reported: 38.4% vs 24.2%, P = 0.029.

Data: per-patient source data (Supplementary Data, sheets Figure 2-6) in `data/khong2025_tacrolimus_rct_source_data.csv`.
The sheets carry no patient ID, but their Group column is identical in order across all sheets, so rows are joined by position.

Statements about the method and protocol are quoted from the paper and its Supplementary Information (trial protocol).

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import mannwhitneyu

d = pd.read_csv("../data/khong2025_tacrolimus_rct_source_data.csv")
ppm, soc = d[d.Group == "PPM"], d[d.Group == "Ctrl"]
rng = np.random.default_rng(0)

def boot_diff(a, b, B=10000):
    a, b = np.asarray(a), np.asarray(b)
    v = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(B)]
    return np.percentile(v, [2.5, 97.5])

ROWS = {}
for col, label in [("%Days >2", "% days >2 ng/mL off target (primary)"), ("ADDTotal", "avg daily deviation, total"),
                   ("ADDBelow", "avg daily deviation below range"), ("ADDAbove", "avg daily deviation above range"),
                   ("Mean TTL", "mean trough level (ng/mL)"), ("LOHosp", "length of stay (days)"), ("DAST", "days to normal AST")]:
    a, b = ppm[col].dropna(), soc[col].dropna()
    p = mannwhitneyu(a, b, alternative="two-sided").pvalue
    lo, hi = boot_diff(a, b)
    ROWS[label] = (a.mean(), b.mean(), a.mean() - b.mean(), lo, hi, p)
    print(f"{label:38s} PPM {a.mean():6.2f}  SOC {b.mean():6.2f}  diff {a.mean() - b.mean():6.2f} [{lo:6.2f}, {hi:6.2f}]  Wilcoxon P = {p:.4f}")

% days >2 ng/mL off target (primary)   PPM   0.24  SOC   0.38  diff  -0.14 [ -0.26,  -0.02]  Wilcoxon P = 0.0591
avg daily deviation, total             PPM   1.40  SOC   1.81  diff  -0.41 [ -0.84,   0.02]  Wilcoxon P = 0.0866


avg daily deviation below range        PPM   0.76  SOC   1.40  diff  -0.64 [ -1.03,  -0.25]  Wilcoxon P = 0.0096


avg daily deviation above range        PPM   0.61  SOC   0.42  diff   0.19 [ -0.09,   0.46]  Wilcoxon P = 0.3309


mean trough level (ng/mL)              PPM   8.41  SOC   7.63  diff   0.78 [ -0.03,   1.56]  Wilcoxon P = 0.0344


length of stay (days)                  PPM  13.11  SOC  17.76  diff  -4.65 [-10.33,   1.46]  Wilcoxon P = 0.0026
days to normal AST                     PPM   7.13  SOC   8.75  diff  -1.62 [ -4.13,   1.67]  Wilcoxon P = 0.0139


## 1. The trial did not test CURATE.AI's quadratic model

- The protocol (Supplementary Information) planned a **"Parabolic Response Surface (PRS)"**, the quadratic dose-response model CURATE.AI is known for.
- The paper reports that the three-day calibration needed for that model ("Function 1") **"could therefore not be applied to this study"**, because of rapid changes after transplant.
- Doses were instead set with "Function 2": the assumption that tomorrow's trough-to-dose ratio equals today's, TTL(t+1)/c(t+1) = TTL(t)/c(t).
  That is a **proportional dose adjustment**, the classic rule of therapeutic drug monitoring for drugs with roughly linear kinetics.

So the trial tests "proportional day-to-day adjustment by a dedicated team" against "clinician dosing", not the per-patient quadratic.

## 1b. Does the primary P value reproduce?

The Figure 2 legend says: "A two-tailed Wilcoxon Rank Sum test was used to compare the groups" (P = 0.029). Re-running candidate tests on the published per-patient values:

In [2]:
from scipy import stats
a, b = ppm["%Days >2"].to_numpy(), soc["%Days >2"].to_numpy()
PRIMARY = {"two-sided Wilcoxon rank-sum (stated in legend)": stats.mannwhitneyu(a, b).pvalue,
           "one-sided Wilcoxon rank-sum": stats.mannwhitneyu(a, b, alternative="less").pvalue,
           "Welch t-test": stats.ttest_ind(a, b, equal_var=False).pvalue,
           "Student t-test": stats.ttest_ind(a, b).pvalue}
for k, v in PRIMARY.items():
    print(f"{k:48s} P = {v:.4f}")
print("tied values in the primary outcome:", len(a) + len(b) - len(np.unique(np.concatenate([a, b]))))

two-sided Wilcoxon rank-sum (stated in legend)   P = 0.0591
one-sided Wilcoxon rank-sum                      P = 0.0296
Welch t-test                                     P = 0.0294
Student t-test                                   P = 0.0310
tied values in the primary outcome: 29


**Reading it.** The published P = 0.029 matches a Welch t-test or a one-sided rank test, not the two-sided rank test the legend names, which gives P = 0.059.
The reported 95% CI (-26.7 to -1.5%) is also the t-based interval. The outcome is a skewed percentage with many ties, which is why a rank test would be the appropriate choice.
The paper's modified intention-to-treat analysis (P = 0.0082) cannot be checked: data for the excluded patients are not published.
The other reported P values (0.0096, 0.087, 0.034, 0.0026, 0.014) reproduce with the stated test. The above-range deviation in the PPM group computes to 0.61 (paper 0.64) with P = 0.33 (paper 0.13).

## 2. Where the benefit comes from: under-dosing in the control arm

In [3]:
fig = go.Figure()
for col, name in (("ADDBelow", "below range"), ("ADDAbove", "above range")):
    fig.add_trace(go.Box(y=ppm[col], name=f"PPM: {name}", boxpoints="all", jitter=0.4))
    fig.add_trace(go.Box(y=soc[col], name=f"SOC: {name}", boxpoints="all", jitter=0.4))
fig.update_layout(yaxis_title="average daily deviation (ng/mL per day)", height=430, showlegend=False,
                  title="PPM cut time below range; time above range went up")
fig

**Reading it.** PPM halved the deviation below range (P about 0.01) while the deviation above range rose (not significant), and mean trough moved from 7.6 to 8.4 ng/mL.
Most of the effect is that standard care ran low. Any rule that aimed at the middle of the range more consistently might have done the same; the trial cannot separate "the method" from "aiming higher".

## 3. Other things to weigh

In [4]:
print("length of stay, median: PPM", ppm.LOHosp.median(), "SOC", soc.LOHosp.median(), "| longest stays:", sorted(d.LOHosp)[-3:])
print("days to normal AST, median: PPM", ppm.DAST.median(), "SOC", soc.DAST.median(), f"(n {ppm.DAST.notna().sum()} vs {soc.DAST.notna().sum()})")

length of stay, median: PPM 10.0 SOC 15.0 | longest stays: [45, 53, 58]
days to normal AST, median: PPM 6.0 SOC 8.5 (n 23 vs 24)


1. **Allocation could be guessed.** Fixed blocks of 4, sequence generated by the principal investigator, clinical team not blinded (Methods; the authors list this as a limitation). In fixed blocks the last allocation of each block is predictable.
2. **Liver recovery differed.** AST normalised faster in the PPM group (6 vs 8.5 days) and stays were shorter (median 10 vs 15 days). The paper links these to dosing; they could equally reflect healthier grafts in one arm by chance, which a 56-patient trial cannot rule out.
3. **One listed secondary outcome has no reported result.** "Percent days outside-of-target range" is named as a secondary outcome, but no value for it was found in the main text or the source data.
4. **Primary outcome matches the registry.** ClinicalTrials.gov lists the same primary measure, "Percentage of Days Far (> 2 ng/mL) Out of Range" (checked 2026-11-09).
5. **Size.** 56 patients, single centre. The bootstrap interval for the primary difference is wide (see the table above).

## Verdict

With the test the paper names, the primary result is P = 0.059, not 0.029. At best a modest, single-centre result for *structured proportional dose adjustment* of tacrolimus, mainly by avoiding under-dosing. It is not evidence for CURATE.AI's per-patient quadratic model, which the trial planned but did not use.